## Imports

In [ ]:
# importing sys
import sys
# This is how you import modules that exist outside of working directory 
# adding mosaique to the system path
sys.path.insert(1, '/workspaces/QML-QPF/mosaiQue') 
sys.path.insert(1, '/workspaces/QML-QPF/Quantifying_Entanglement')

import mosaique as mq
import scipy
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import numpy as np
import pennylane as qml
import os
from mosaique.models.operation import OperationLayer
from tensorflow import keras
import tensorflow as tf 
import time
from Entropy import compute_entropy_for_pair_batch

## Check GPU installation is working

In [ ]:
import tensorrt as trt

print(f' tensorflow version {tf.__version__}')

print(f' tensorrt version {trt.__version__}')

print(tf.config.list_physical_devices('GPU'))

In [ ]:

# Set the environment for asynchronous GPU usage
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

In [ ]:
def operation():
    dev = qml.device("default.qubit.tf", wires=4)
    @qml.qnode(dev, interface='tf')
    def cnot(inputs):
        inputs = inputs * np.pi
        qml.AngleEmbedding(inputs[:,...], wires=range(4), rotation='Y')

        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[2, 3])

        # Measurement producing 4 classical output values
        return [qml.expval(qml.PauliZ(j)) for j in range(4)]
    return cnot


In [ ]:
def preset():
    # load the data
    mnist_dataset = keras.datasets.mnist
    
    # name of the folder where will save the filtered images for training
    tr_layer = mq.ConvolutionLayer4x4("_OA_results_Mnist_train_with_val")
    # name of the folder where will save the filtered images for testing
    te_layer = mq.ConvolutionLayer4x4("_OA_results_Mnist_test_with_val")
    
    
    (tr_images, tr_labels), (te_images, te_labels) = mnist_dataset.load_data()
    
   
    tr_layer.fit(tr_images)
    te_layer.fit(te_images)
    tr_images = tr_layer.transform(tr_images)
    te_images = te_layer.transform(te_images)
    return ((tr_layer, (tr_images, tr_labels)), (te_layer,(te_images, te_labels)))

with ProcessPoolExecutor() as executor:
    future = executor.submit(preset)

((train_layer, (train_images, train_labels)), (test_layer,(test_images, test_labels))) = future.result() #This blocks until the task completes

In [ ]:

permutations = np.asarray(list(itertools.permutations(range(4))))

def pool(x, call, p, l):
    op = OperationLayer(call())
    predict = l.post_transform(op.pre_op.predict(x,batch_size=1000))
    l.save(predict, p)

In [ ]:
for j in range(3):
    with ProcessPoolExecutor(8) as executor:
        runner = {
            executor.submit(pool, x=train_images[:,:,p], call=operation, p=p, l=train_layer): p for p in permutations[8*j:8*(j+1)]
        }
        for future in as_completed(runner):
            runner.pop(future)

In [ ]:
for j in range(3):
    with ProcessPoolExecutor(8) as executor:
        runner = {
            executor.submit(pool, x=test_images[:,:,p], call=operation, p=p, l=test_layer): p for p in permutations[8*j:8*(j+1)]
        }
        for future in as_completed(runner):
            runner.pop(future)

## Filtering Data by Entropy degree

In [ ]:
train_images.shape

In [ ]:
permutations[12]

In [ ]:
train_layer.name

In [ ]:
input_system_03 = np.array([[2,3],[1,5],[8,3],[1,9],]) 
input_system_21 = np.array([[20,30],[7,58],[24,34],[15,54],])
# Assuming the same slicing  input_system_03 = input_data[:, :, :2]

task_inputs = list(zip(input_system_03, input_system_21))
task_inputs

In [ ]:
input1_batch, input2_batch = task_inputs

In [ ]:
print(input1_batch, input2_batch)

In [ ]:
# --- Function to process a single input type ---
def process_input_type(input_data, config, entropy_dict_for_symm):
    """
    Helper function to encapsulate the parallel processing logic for one input type.
    """
    input_system_03 = input_data[:, :, :2]
    input_system_21 = input_data[:, :, 2:4] # Assuming the same slicing  input_system_03 = input_data[:, :, :2]

    task_inputs = list(zip(input_system_03, input_system_21))
    print("Im inside process_input_type")
    print("len of task inputs is ", len(task_inputs) )
    # print(f"\nStarting parallel entropy computation for {data_name} data ({len(task_inputs)} batches)...")
    type_start_time = time.time()
    # from collections import defaultdict

    # entropy_dict_for_symm = defaultdict(list)   # missing keys become []
    # config = ''.join(map(str, config)) 
    with ProcessPoolExecutor(max_workers=None) as executor:
        results_iterator = executor.map(compute_entropy_for_pair_batch, task_inputs)

        for i, result in enumerate(results_iterator):
            print(f"i is {i} and result is {type(result)} len {len(result)}")
            (avg_s1_rho0, avg_s1_rho1), \
            (avg_s2_rho0, avg_s2_rho1), \
            (max_s1_rho0, max_s1_rho1), \
            (max_s2_rho0, max_s2_rho1), \
            (S1_rho_0), \
            (S1_rho_1), \
            (S2_rho_0), \
            (S2_rho_1) = result
            
            # Append to the lists directly from the dictionary
            # Using dictionary keys here
            entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'].append((avg_s1_rho0, avg_s1_rho1))
            entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2'].append((avg_s2_rho0, avg_s2_rho1))
            entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'].append((max_s1_rho0, max_s1_rho1))
            entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2'].append((max_s2_rho0, max_s2_rho1))
            # New ones to capture pair entanglement for subsequent filtering
            entropy_dict_for_symm['S1_rho_0_' + config + '_Entropy_S1'].append(S1_rho_0)
            entropy_dict_for_symm['S1_rho_1_' + config + '_Entropy_S2'].append(S1_rho_1)
            entropy_dict_for_symm['S2_rho_0_' + config + '_Entropy_S1'].append(S2_rho_0)
            entropy_dict_for_symm['S2_rho_1_' + config + '_Entropy_S2'].append(S2_rho_1)

            if (i + 1) % 100 == 0:
                print(f"Processed {i + 1}/{len(task_inputs)} batches for {config}.")
                
        
        # Convert lists to NumPy arrays
        entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'] = np.array(entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'])
        entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2'] = np.array(entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2'])
        entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'] = np.array(entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'])
        entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2'] = np.array(entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2'])
        # New ones to capture pair entanglement for subsequent filtering
        entropy_dict_for_symm['S1_rho_0_' + config + '_Entropy_S1']  = np.array(entropy_dict_for_symm['S1_rho_0_' + config + '_Entropy_S1'])    
        entropy_dict_for_symm['S1_rho_1_' + config + '_Entropy_S2']  = np.array(entropy_dict_for_symm['S1_rho_1_' + config + '_Entropy_S2'])
        entropy_dict_for_symm['S2_rho_0_' + config + '_Entropy_S1']  = np.array(entropy_dict_for_symm['S2_rho_0_' + config + '_Entropy_S1'])
        entropy_dict_for_symm['S2_rho_1_' + config + '_Entropy_S2']  = np.array(entropy_dict_for_symm['S2_rho_1_' + config + '_Entropy_S2'])
        
        # compute the total entry of the entire system S1 + S2
        # Compute the total entropy of the entire system S1 + S2
        # Ensure avg_s1_array and avg_s2_array are in the correct shape (N, 2)
        # The output of compute_entropy_for_pair_batch is (rho0, rho1) tuples, which will become (N, 2) arrays.
        entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1_S2'] = entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'] \
                                                                            + entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2']
        entropy_dict_for_symm[f'Total_Max_{config}_Entropy_S1_S2'] = entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'] \
                                                                            + entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2']

    # --- Saving Data as CSV ---
    # 1. Create the data_name specific folder

    entropy_folder_name = train_layer.name + f"/{config}/Entropy"
    # train
    # entropy_folder_name = "_OA_train"
    output_folder_path = os.path.join(entropy_folder_name) # Replace spaces for folder names
    os.makedirs(output_folder_path, exist_ok=True)
    print(f"Created/Ensured directory: {output_folder_path}")

    # 2. Save each list/array corresponding to each key as a CSV file
    for key, value in entropy_dict_for_symm.items():
        if isinstance(value, np.ndarray): # Only save if it's a NumPy array
            file_path = os.path.join(output_folder_path, f"{key}.csv")
            # Use fmt='%.8f' for float formatting, delimiter=',' for CSV
            np.savetxt(file_path, value, delimiter=',', fmt='%.8f')
            print(f"Saved {key}.csv to {output_folder_path}")
        else:
            print(f"Skipping saving for key '{key}' as it's not a NumPy array (type: {type(value)})")
    type_end_time = time.time()
    print(f"Finished parallel computation for data for config {config} in {type_end_time - type_start_time:.2f} seconds.")

In [ ]:
input_system_03 = input_data[:, :, :2]
input_system_21 = input_data[:, :, 2:4] # Assuming the same slicing for all input types

task_inputs = list(zip(input_system_03, input_system_21))


In [ ]:
# Scaling data
scaled_sample = train_images
# Encoding data using angle encoding
input = scaled_sample * np.pi

In [ ]:
input.shape

In [ ]:
input[0,0]

In [ ]:
input_data = input
input_system_03 = input_data[:, :, :2]
input_system_21 = input_data[:, :, 2:4] 
task_inputs = list(zip(input_system_03, input_system_21))

In [ ]:
task_inputs[1][1].shape

In [ ]:
input1_batch, input2_batch = task_inputs

In [ ]:
len(task_inputs)

In [ ]:
train_images[0]

In [ ]:
# Scaling data
scaled_sample = train_images/255.
# Encoding data using angle encoding
input = scaled_sample * np.pi
from collections import defaultdict

for config in permutations:
    # rearranging the input in respect to the configs
    input = input[:,:,config]

    entropy_dict_for_symm = defaultdict(list)   # missing keys become []
    config = ''.join(map(str, config))
    process_input_type(input, config, entropy_dict_for_symm)
    # break

In [ ]:
permutations[-3:]

In [ ]:
# Scaling data
scaled_sample = train_images/255.
# Encoding data using angle encoding
input = scaled_sample * np.pi
from collections import defaultdict

for config in permutations[-3:]:
    # rearranging the input in respect to the configs
    input = input[:,:,config]

    entropy_dict_for_symm = defaultdict(list)   # missing keys become []
    config = ''.join(map(str, config))
    process_input_type(input, config, entropy_dict_for_symm)
    # break

In [ ]:
# --- Function to process and print statistics from a CSV file ---
def analyze_entropy_file(file_path):
    try:
        # Load the data from the CSV file
        data = np.loadtxt(file_path, delimiter=',')

        # Handle cases where data might be a single value or 1D array
        if data.ndim == 0: # A single number
            data = np.array([data])
        elif data.ndim == 1: # A single column or row
            # If it's (N,), mean/var will be single values.
            # If it's (N, 2), it will remain (N, 2).
            pass # np.mean and np.var handle 1D arrays correctly for axis=0

        # Compute average (mean) along axis=0 (for each column)
        avg_values = np.mean(data, axis=0)
        # Compute variance along axis=0 (for each column)
        var_values = np.var(data, axis=0)
        # Compute Standard Deviation along axis=0 (for each column)
        std_values = np.std(data, axis=0)
        

        # print(f"\n--- Analysis for: {data_name} / {symmetry_name} / {entropy_key} ---")
        print(f"File: {file_path}")
        print(f"Data Shape: {data.shape}")
        print(f"Average (Mean) per column: {avg_values}")
        # print(f"Variance per column: {var_values}")
        # print(f"Standard Deviation per column: {std_values}")
        return avg_values

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

In [ ]:
for config in permutations:
    # rearranging the input in respect to the configs
    input = input[:,:,config]
    config = ''.join(map(str, config))
    entropy_folder_name = train_layer.name + f"/{config}/Entropy/Total_Avg_{config}_Entropy_S1_S2.csv"
    avg_entropy = analyze_entropy_file(entropy_folder_name)
    
    

In [ ]:
train_layer.open(permutations[0]).shape

In [ ]:
import numpy as np

file_path = "/workspaces/QML-QPF/Entanglement_Filtering/_OA_results_Mnist_train_with_val/0123/Entropy/S1_rho_0_0123_Entropy_S1.csv"
data = np.loadtxt(file_path, delimiter=',')

In [ ]:
data.shape

In [ ]:

# Check if any element is greater than 2
# condition_array = a > 2 
# condition_array is now a boolean array: [False, False, False, True, True]

# result = condition_array.any() 
for i in range(len(input[0])):
    for el in input[0][i]:
        if el - 0 > 0.0001:
            print(i)


In [ ]:
input[0][34]

In [ ]:
i = 20
s = 20
data[1][i:i+s]

In [ ]:
data[0][34]

We will focus only in the best 3 configs.

In [ ]:
D = {"Configs": { "P_diagonal_config" : np.asarray([0,3,2,1]),
                                    "P_vertical_config" : np.asarray([1,0,3,2]),
                                    "P_horizontal_config" : np.asarray([1,3,2,0])
                                  }
}

In [ ]:
Diagonal = np.asarray([0,3,2,1])
filtered_input = train_layer.open(Diagonal)
reshaped_filtered_input = train_layer.transform(filtered_input)


In [ ]:
filtered_input.shape

In [ ]:
reshaped_filtered_input = train_layer.transform(filtered_input)

In [ ]:
reshaped_filtered_input[0]

In [ ]:
reshaped_filtered_input[0,2,:] == filtered_input[0,0,2,:]

In [ ]:
#S1_rho_0_0321_Entropy_S1 : S1_rho_0 = system 1;Entropy_S1 = qubit 1

file_path = "/workspaces/QML-QPF/Entanglement_Filtering/_OA_results_Mnist_train_with_val/0321/Entropy/S1_rho_0_0321_Entropy_S1.csv"
S1_rho0 = np.loadtxt(file_path, delimiter=',')
file_path = "/workspaces/QML-QPF/Entanglement_Filtering/_OA_results_Mnist_train_with_val/0321/Entropy/S2_rho_0_0321_Entropy_S1.csv"
S2_rho0 = np.loadtxt(file_path, delimiter=',')
# File namne to store the pooled sample
Entanglement_pooled_file_name = f"{train_layer.name}/{''.join(map(str,Diagonal))}/Entanglement_pooled_Dgl_{''.join(map(str,Diagonal))}"  


In [ ]:
Entanglement_pooled_file_name = f"{train_layer.name}/{''.join(map(str,Diagonal))}/Entanglement_pooled_Dgl_{''.join(map(str,Diagonal))}"  
Entanglement_pooled_file_name

In [ ]:
S2_rho0.shape

In [ ]:
a = np.array([],)

In [ ]:
np.append(a, [1,3])

In [ ]:
l = []

In [ ]:
type(l)

In [ ]:
l.extend([1,3])

In [ ]:
l

### How to flatten a list of lists of different length


In [ ]:
import joblib
import gc



In [ ]:
sample_size = reshaped_filtered_input.shape[0]  # number of images
Entanglement_pooled_sample = list()
threshold = 0.106 # The mean of the entire dataset--will update it accordingly
for i in range(sample_size):
    Entanglement_pooled_img = list()
    for j in range(S1_rho0.shape[-1]):
        #
        tmp_list = list()
        # Checking for S1
        if S1_rho0[i,j] < threshold: # keeping both qubits
            tmp_list.extend(reshaped_filtered_input[i,j,:2]) 
        else: # Keeping only target qubit
            tmp_list.append(reshaped_filtered_input[i,j,1])
            
        # Checking for S2
        if S2_rho0[i,j] < threshold: # keeping both qubits
            tmp_list.extend(reshaped_filtered_input[i,j,2:]) 
        else: # Keeping only target qubit
            tmp_list.append(reshaped_filtered_input[i,j,-1])
        # appending the pooled filtered input to the image
        Entanglement_pooled_img.append(tmp_list)
    # appending the pooled filtered image to the sample
    Entanglement_pooled_sample.append(Entanglement_pooled_img)
    
joblib.dump(Entanglement_pooled_sample, Entanglement_pooled_file_name, compress=3)

# free memory so the notebook stays fast
del Entanglement_pooled_sample
gc.collect()
        

In [ ]:
l=32
reshaped_filtered_input[0,l]

In [ ]:
reshaped_filtered_input

In [ ]:
len(Entanglement_pooled_sample[0])

In [ ]:
from matplotlib import pyplot as plt
post = filtered_input
_min, _max = np.amin(post), np.amax(post)
# _min1, _max1 = np.amin(post1), np.amax(post1)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Plot all output channels for quantum cnot
for c in range(4):
    axes[c].imshow(post[0,:,:,c],vmin = _min, vmax = _max)

### Row-Level Padding

In [ ]:
import numpy as np

def pad_jagged_image(image_list, max_c=4):
    # Create a fixed-size buffer of zeros
    padded_img = np.zeros((196, max_c))
    
    for i, row in enumerate(image_list):
        # row is a 1D array of length 2, 3, or 4
        length = len(row)
        padded_img[i, :length] = row
        
    return padded_img


In [ ]:
padded_S = [0]*4

In [ ]:
padded_S

In [ ]:
r = [1,2]
length = len(r)
padded_S[:length] = r
print(padded_S)

In [ ]:
len(Entanglement_pooled_sample[0])

In [ ]:
padded_Entanglement_pooled_sample = pad_jagged_image(Entanglement_pooled_sample[0])

In [ ]:
Entanglement_pooled_sample[0]

In [ ]:
padded_Entanglement_pooled_sample

## Trainign model


In [ ]:
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# change the subfolder name to save your data in different location based on what NN model you used
# it's a convenient way to avoid losing your training data of each model
# Here are some preset examples you can use
# 1 FC layer
run_subfolder = "_1FC"
# run_subfolder = "_2nodes_2FC"
# # 2 FC layer
# run_subfolder = "_2FC"
# # 3 FC layer
# run_subfolder = "_3FC"
# # 4 FC layer
# run_subfolder = "_4FC"
# early stopping epochs -- good practice between 10-20
patience = 15

# taking the measurement of only target qubits [[1,3]]
Entanglement_pool_folder_name = "Entanglement_pool/"
run_subfolder += "/" + Entanglement_pool_folder_name

def run(tr_images, label):
    log_dir = train_layer.name + "/run" + run_subfolder +  "/"
    log_dir += str(patience) +  "/"
    print(' log_dir ', log_dir)
    log_file =  log_dir + label
    # print(' log_file ', log_file)
    os.makedirs(log_dir, exist_ok=True)
    
    csv_logger = CSVLogger(log_file)
    
    
    # tensorboard_callback = keras.callbacks.TensorBoard(
    #     log_dir=log_file,
    #     histogram_freq=1,
    #     write_graph=True,
    #     write_images=True,
    #     write_steps_per_second=True,
    #     update_freq='batch',
    #     profile_batch=1,
    #     embeddings_freq=1,
    #     embeddings_metadata=None
    # )
    
    number_of_classes = len(np.unique(test_labels))
    # Chose which model you want to use depending on the number of hidden layers
    # Uncomment your model and run the code--pay attention to your folder/file naming to keep everything organized 
    ####---------------------------------------------------------------------####
    
    # # simple ANN model with 0 hidden layers: 1FC
    # q_model = keras.models.Sequential([
    #     keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
    #     keras.layers.Flatten(),
    #     keras.layers.Dense(number_of_classes, activation="softmax")
    # ])
    
    ####---------------------------------------------------------------------####
    
    # # simple ANN model with 1 hidden layers: 2FC
    # q_model = keras.models.Sequential([
    #     keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
    #     keras.layers.Flatten(),
    #     keras.layers.Dense(512, activation='relu'),
    #     keras.layers.Dense(number_of_classes, activation="softmax")
    # ])
    
    ####---------------------------------------------------------------------####
    
    # simple ANN model with 2 hidden layers: 3FC
    q_model = keras.models.Sequential([
        keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dense(number_of_classes, activation='softmax')
    ])
    
    ####---------------------------------------------------------------------####
    
    # # simple ANN model with 3 hidden layers: 4FC
    # q_model = keras.models.Sequential([
    #     keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
    #     keras.layers.Flatten(),
    #     keras.layers.Dense(512, activation='relu'),
    #     keras.layers.Dense(256, activation='relu'),
    #     keras.layers.Dense(128, activation='relu'),
    #     keras.layers.Dense(number_of_classes, activation='softmax')
    # ])
    
    ####---------------------------------------------------------------------####
    
    q_model.compile(
        optimizer='adam',
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    
    # 
    batch_size = 128
    epochs = 60
    
    
    # Stop training when a monitored metric has stopped improving.
    Early_stop = EarlyStopping(monitor="val_accuracy",
                                min_delta=0, # Minimum change in the monitored quantity to qualify as an improvement, i.e. an absolute change of less than min_delta, will count as no improvement.
                                patience=patience, # Number of epochs with no improvement after which training will be stopped.
                                verbose=1, # 1 displays messages when the callback takes an action
                                mode="max", # will stop when the quantity monitored has stopped increasing
                                baseline=None,
                                restore_best_weights=True, # to restore model weights from the epoch with the best value of the monitored quantity
                                start_from_epoch=0, #  Number of epochs to wait before starting to monitor improvement.
                            )

    
    # Callback to save the Keras model or model weights at some frequency.
    checkpoint_filepath = log_file  + 'model.keras'
    model_checkpoint_callback = ModelCheckpoint(filepath=checkpoint_filepath,
                                                monitor='val_accuracy',
                                                mode='max',
                                                verbose=1,
                                                save_best_only=True
                                                )
    # Reduce learning rate when a metric has stopped improving.
    reduce_lr_acc = ReduceLROnPlateau(monitor='val_accuracy',
                                        factor=0.1, # Float. Factor by which the learning rate will be reduced. new_lr = lr * factor.
                                        patience=10, # Integer. Number of epochs with no improvement after which learning rate will be reduced.
                                        verbose=1, # 1: update messages.
                                        mode="max", # 'max' mode it will be reduced when the quantity monitored has stopped increasing
                                        min_delta=0.0001, # Float. Threshold for measuring the new optimum, to only focus on significant changes.
                                        cooldown=0, #  Integer. Number of epochs to wait before resuming normal operation after the learning rate has been reduced.
                                        min_lr=0.0 #  Float. Lower bound on the learning rate.
                                    )


    
    q_history = q_model.fit(
        tr_images,
        train_labels,
        # validation_data=(te_images, test_labels),
        validation_split=0.2,
        batch_size=batch_size,
        epochs=epochs,
        verbose=2,
        # callbacks=[tensorboard_callback]
        callbacks = [Early_stop,csv_logger,model_checkpoint_callback, reduce_lr_acc]
    )

def model(variant, tr_layer):
    tr_images = tr_layer.open(variant)
    # to get only the target qubits
    tr_images = tr_images[:,:,:,[1,3]]
    # te_images =  te_layer.open(variant)

    label = ''.join(map(str,variant))

    run(tr_images, label)